# 09o Does the personal gain shrink as the crowd grows? (THINGS-EEG1 development people)

**Kaggle:** CPU, Internet on. **Inputs:** the outputs of 08a and 08b (only the 22 development people are opened).

Development analysis, not a prespecified test. Personalization gains can shrink as the population model gets stronger (Tao and Chen 2026). Here borrowing is
built from n of the 21 other people (n = 2, 5, 10, 15, 21; five random subsets for each n below 21) and the personal
map is fitted on top at k = 250 and 500 (draw 13). An exchanged control applies a map fitted for another person.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '09o-sourcecount', 'cpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"n_sources": [2, 5, 10, 15, 21], "subsets_per_n": 5, "ks": [250, 500], "draw": 13, "donors": 3, "window_s": [0.1, 0.2], "map_window_ms": 100, "seed": 39}')
PEOPLE = ['sub-04', 'sub-07', 'sub-11', 'sub-12', 'sub-13', 'sub-14', 'sub-15', 'sub-17', 'sub-20', 'sub-21', 'sub-24', 'sub-25', 'sub-26', 'sub-29', 'sub-30', 'sub-34', 'sub-35', 'sub-38', 'sub-39', 'sub-46', 'sub-48', 'sub-49']

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker
from scipy import stats

from cross_subject_encoding.data import things_eeg2 as te
from cross_subject_encoding.eval import eeg1_h4b as h4b
from cross_subject_encoding.eval import h4b_analysis as ha
from cross_subject_encoding.eval.paper_correlation import paper_correlation
from cross_subject_encoding.models import paper_linearizing as pl
from cross_subject_encoding.utils import config as C
from cross_subject_encoding.utils.kaggle_export import export_to_dataset

from cross_subject_encoding.data import things_eeg1 as e1
from cross_subject_encoding.utils.eeg1_pilot import sha256_file, source_file
assert sha256_file(C.find_artifact("eeg1_paper_kpca1000.npz")) == "61d36682db158fc622efe721c7759f70f52868a828aa9a66d395fdff7318ec1f"
with np.load(C.find_artifact("eeg1_paper_kpca1000.npz")) as f:
    z_train, z_test = f["train"].astype(np.float64), f["test"].astype(np.float64)
subsets = json.loads(C.find_artifact("eeg1_k_subsets.json").read_text())
people = sorted(PEOPLE)  # the 22 sources only; the 24 confirmation targets are never opened
y_train, bio_test = {}, {}
for s in people:
    for part in ("training", "test"):
        rel = te.load_part(source_file(Path("/kaggle/input"), s, part, people))
        if part == "training":
            y_train[s], times = rel["data"][:, 0], rel["times"]
        else:
            bio_test[s] = rel["data"]
order = lambda s, d, k: np.asarray(subsets["orders"][s][str(d)])[:k]
print(len(people), 'people loaded |', f'{(time.monotonic() - T0) / 60:.1f} min')

In [ ]:
N_TRAIN, N_TEST = len(z_train), len(z_test)
n_ch, n_t = 17, 100
query = np.concatenate([z_train, z_test])
src_train = np.empty((len(people), N_TRAIN, n_ch * n_t), np.float32)
src_test = np.empty((len(people), N_TEST, n_ch * n_t), np.float32)
for i, s in enumerate(people):
    pred = pl.ols_paper(z_train, y_train[s].reshape(N_TRAIN, -1), query)
    src_train[i], src_test[i] = pred[:N_TRAIN], pred[N_TRAIN:]
window = (times >= FIXED["window_s"][0]) & (times < FIXED["window_s"][1])
win = ha.window_index(times, FIXED["map_window_ms"])
t3 = lambda a, n: np.asarray(a, np.float64).reshape(n, n_ch, n_t)
d = FIXED["draw"]
rows = []
for i, person in enumerate(people):
    t = time.monotonic()
    rng = np.random.default_rng([FIXED["seed"], i])
    others = [j for j in range(len(people)) if j != i]
    preds = {}
    for n in FIXED["n_sources"]:
        reps = 1 if n == len(others) else FIXED["subsets_per_n"]
        for r in range(reps):
            sub = others if n == len(others) else sorted(rng.choice(others, n, replace=False).tolist())
            avg_train, avg_t3 = src_train[sub].mean(0), t3(src_test[sub].mean(0), N_TEST)
            preds[f"avg|{n}|{r}|all"] = avg_t3
            for k in FIXED["ks"]:
                idx = order(person, d, k)
                coefs, _ = h4b.fit_map(t3(avg_train[idx], k), np.asarray(y_train[person][idx], np.float64), win, seed=d)
                preds[f"h4b|{n}|{r}|{k}"] = h4b.apply_map(coefs, avg_t3, win)
    # exchanged parameters: a map fitted for another person (donor) on the donor's own k images, with the donor's
    # average built from the 20 sources that are neither the donor nor the target, applied to the target's average of 21
    avg21 = t3(src_test[others].mean(0), N_TEST)
    for r, donor in enumerate(sorted(rng.choice(others, FIXED["donors"], replace=False).tolist())):
        pool = [j for j in others if j != donor]
        donor_avg_train = src_train[pool].mean(0)
        for k in FIXED["ks"]:
            idx = order(people[donor], d, k)
            coefs, _ = h4b.fit_map(t3(donor_avg_train[idx], k), np.asarray(y_train[people[donor]][idx], np.float64), win, seed=d)
            preds[f"exchanged|21|{r}|{k}"] = h4b.apply_map(coefs, avg21, win)
    preds = {name: np.asarray(p).reshape(N_TEST, n_ch, n_t) for name, p in preds.items()}
    if not all(np.isfinite(p).all() for p in preds.values()):
        raise ValueError(f"{person}: nonfinite prediction")
    scored = paper_correlation(preds, bio_test[person], n_iter=100)
    for name, curve in scored["correlation"].items():
        method, n, r, k = name.split("|")
        rows.append({"person": person, "method": method, "n_sources": int(n), "subset": int(r), "k": k,
                     "r_100_200ms": float(curve[:, window].mean())})
    pd.DataFrame(rows).to_csv(C.METRICS_DIR / "09o_scores.csv", index=False)
    print(person, f"{time.monotonic() - t:.0f} s |", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
frame = pd.DataFrame(rows)
def per_person(method, n, k):
    f = frame[(frame.method == method) & (frame.n_sources == n) & (frame.k == (str(k) if method != "avg" else "all"))]
    return f.groupby("person")["r_100_200ms"].mean().reindex(people).to_numpy()
def ci(x):
    se = x.std(ddof=1) / np.sqrt(len(x))
    t = stats.t.ppf(0.975, len(x) - 1)
    return {"mean": float(x.mean()), "lower95": float(x.mean() - t * se), "upper95": float(x.mean() + t * se),
            "positive": int((x > 0).sum()), "n": int(len(x))}
kmean = lambda method, n: np.mean([per_person(method, n, k) for k in FIXED["ks"]], 0)
summary = {"by_n": {}, "post_hoc": "development; the 24 reserved EEG1 people are not opened"}
for n in FIXED["n_sources"]:
    avg, h = per_person("avg", n, None), kmean("h4b", n)
    summary["by_n"][n] = {"avg": float(avg.mean()), "h4b": float(h.mean()), "h4b_minus_avg": ci(h - avg)}
g = {n: kmean("h4b", n) - per_person("avg", n, None) for n in FIXED["n_sources"]}
summary["gain_21_minus_gain_5"] = ci(g[21] - g[5])
summary["gain_21_over_gain_5"] = float(g[21].mean() / g[5].mean())
summary["h4b_minus_exchanged_21"] = ci(kmean("h4b", 21) - kmean("exchanged", 21))
summary["exchanged_minus_avg_21"] = ci(kmean("exchanged", 21) - per_person("avg", 21, None))
summary["h4b_5_minus_avg_21"] = ci(kmean("h4b", 5) - per_person("avg", 21, None))
rho = [stats.spearmanr(np.log(FIXED["n_sources"]), [g[n][p] for n in FIXED["n_sources"]])[0] for p in range(len(people))]
summary["spearman_gain_vs_log_n"] = {"median": float(np.median(rho)), "negative": int(np.sum(np.array(rho) < 0)), "n": len(rho)}
summary["rule"] = ("descriptive: the personal gain holds up as the crowd grows if, at n = 21, H4b minus others' average has a "
                   "95% lower bound above 0 and is at least half its size at n = 5; personal beats exchanged if H4b minus "
                   "exchanged at n = 21 has a 95% lower bound above 0")
(C.METRICS_DIR / "09o_summary.json").write_text(json.dumps(summary, indent=2))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.9))
ns = FIXED["n_sources"]
xs = np.arange(len(ns))  # crowd sizes as evenly spaced categories
ax[0].plot(xs, [summary["by_n"][n]["avg"] for n in ns], "o--", color="#7f7f7f", label="others' average")
ax[0].plot(xs, [summary["by_n"][n]["h4b"] for n in ns], "o-", color="#d62728", label="personal map (k = 250/500)")
ax[0].set(xlabel="source people averaged", ylabel="r, 100-200 ms", title="EEG1, 22 sources: by crowd size")
ax[0].legend(fontsize=8)
for p in range(len(people)):
    ax[1].plot(xs, [g[n][p] for n in ns], color="grey", alpha=0.3, lw=0.8)
ax[1].plot(xs, [g[n].mean() for n in ns], "o-", color="#d62728", lw=2, label="mean")
ax[1].axhline(0, color="black", lw=0.5)
ax[1].set(xlabel="source people averaged", ylabel="personal map minus others' average", title="personal gain by crowd size")
for a in ax:
    a.set_xticks(xs)
    a.set_xticklabels([str(n) for n in ns])
fig.tight_layout()
fig.savefig(C.METRICS_DIR / "09o_figure.png", dpi=150)
print(json.dumps(summary, indent=1)[:3500])

In [ ]:
export_to_dataset(slug="cross-subject-brain-encoding-09o-outputs", title="XSUB 09o source count outputs",
                  files=[C.METRICS_DIR / f"09o_{x}" for x in ("summary.json", "scores.csv", "figure.png")],
                  include_dirs=[], message="personal gain against crowd size")